# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.GORILLA;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
%%sql -r dataframe_2
USE ROLE GORILLA_DATA5035_ROLE; --this is the example from class we had been given. 
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    import math

    dx = x - origin_x
    dy = y - origin_y

    tile_width = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    tile_col = int(dx // tile_width)
    tile_row = int(dy // tile_height)

    tile_col_letter = chr(ord('A') + tile_col)
    tile_row_letter = chr(ord('A') + tile_row)
    tile = tile_row_letter + tile_col_letter

    local_x = dx - tile_col * tile_width
    local_y = dy - tile_row * tile_height

    su_col = int(local_x // su_size)
    su_row = int(local_y // su_size)

    su_row_from_top = (tile_grid_y - 1) - su_row
    su = int(su_row_from_top * tile_grid_x + su_col + 1)

    subcell_size = su_size / subcell_grid
    sc_local_x = local_x - su_col * su_size
    sc_local_y = local_y - su_row * su_size

    sc_col = int(sc_local_x // subcell_size)
    sc_row = int(sc_local_y // subcell_size)

    sc = int(sc_row * subcell_grid + sc_col + 1)

    return {'tile': tile, 'survey_unit': su, 'subcell': sc}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

This is where we get into the exercise and homework. 

## Compute Layered Averages

In [ ]:
%%sql -r Aggregate
CREATE OR REPLACE VIEW COMBINED_SURVEY_VIEW AS
WITH base AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata --this pulls the data from the database of our professor
),
grid_labeled AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        grid['tile']::STRING         AS tile,
        grid['survey_unit']::INTEGER AS survey_unit,
        grid['subcell']::INTEGER     AS subcell
    FROM base
),
full_stats AS (
    SELECT
        AVG(READING)    AS full_avg,
        STDDEV(READING) AS full_stddev
    FROM data5035.spring26.sdg_001_ra226_scandata
),
tile_stats AS (
    SELECT
        tile,
        AVG(READING)   AS tile_avg,
        COUNT(*)            AS reading_count
    FROM grid_labeled
    GROUP BY tile
)
SELECT
    g.EASTING,
    g.NORTHING,
    g.READING,
    g.tile,
    g.survey_unit,
    g.subcell,

    -- this is for survey units
    RANK() OVER (
        PARTITION BY g.tile, g.survey_unit
        ORDER BY g.READING DESC
    ) AS subcell_rank_in_su,

    -- this does the ranking for the tiles
    RANK() OVER (
        PARTITION BY g.tile
        ORDER BY g.READING DESC
    ) AS su_rank_in_tile,

    -- for creating a stability score 
    1.0 / (1.0 + STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit)) AS stability_score,

    -- this helps set up a Sensor Variability Index
    STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit) /
        NULLIF(AVG(g.READING) OVER (PARTITION BY g.tile, g.survey_unit), 0) AS sensor_variability_index,

    -- This uses the gap between the zscore and mean. This is a normal formula. 
    (t.tile_avg - gs.full_avg) / NULLIF(gs.full_stddev, 0) AS z_score,

    -- This is for referencing the range status with specific cases based off of when to alarm or not
    CASE
        WHEN g.READING < 5   THEN 'OK'
        WHEN g.READING < 7.4 THEN 'Warning' --chosen from the prompt
        ELSE                           'Alarm'
    END AS reference_range_status,

  --this is the portion for multiple layers that I got assistance on as well. 

    -- subcells
    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit, g.subcell
    ) AS avg_sensor_subcell,

    -- survey units
    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit
    ) AS avg_sensor_survey_unit,

    -- for the tiles
    AVG(g.READING) OVER (
        PARTITION BY g.tile
    ) AS avg_sensor_tile,

    -- This finds the average
    AVG(g.READING) OVER () AS avg_sensor_full

FROM grid_labeled g
JOIN tile_stats t ON g.tile = t.tile
CROSS JOIN full_stats gs;


In [ ]:
%%sql -r dataframe_14
SELECT *
FROM COMBINED_SURVEY_VIEW ORDER BY z_score DESC
LIMIT 10;

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE VIEW REFERENCE_VIEW AS
SELECT
    grid['tile']::STRING         AS tile,
    grid['survey_unit']::INTEGER AS survey_unit,
    grid['subcell']::INTEGER     AS subcell,
    READING,

    -- This controls the reading settings. 
    CASE
        WHEN READING < 5   THEN 'OK'
        WHEN READING < 7.4 THEN 'Warning'
        ELSE                     'Alarm'
    END AS reference_status

FROM (
    SELECT
        READING,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata
);

In [ ]:
%%sql -r dataframe_13
SELECT *
FROM REFERENCE_VIEW
ORDER BY SENSOR_VALUE DESC
LIMIT 10;

SELECT
    reference_status,
    COUNT(*) AS reading_count
FROM REFERENCE_VIEW
GROUP BY reference_status
ORDER BY reading_count DESC;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

This is done lower in the code. 

In [ ]:
%%sql -r dataframe_11
USE ROLE GORILLA_DATA5035_ROLE;
CREATE OR REPLACE FUNCTION DATA5035.GORILLA.CONVERT_XY(
    X FLOAT,
    Y FLOAT,
    ORIGIN_X FLOAT,
    ORIGIN_Y FLOAT,
    SU_SIZE FLOAT,
    TILE_GRID_X NUMBER(38,0),
    TILE_GRID_Y NUMBER(38,0),
    SUBCELL_GRID NUMBER(38,0)
)
RETURNS OBJECT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
HANDLER = 'convert_xy'
AS 
$$
def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    dx = x - origin_x
    dy = y - origin_y

    tile_width = su_size * tile_grid_x
    tile_height = su_size * tile_grid_y

    # --- TILE INDEX ---
    tile_col = int(dx // tile_width)
    tile_row = int(dy // tile_height)

    # Column letter first (matches 'AA', 'AB', etc.)
    tile = chr(ord('A') + tile_col) + chr(ord('A') + tile_row)

    # --- LOCAL POSITION INSIDE TILE ---
    local_x = dx - tile_col * tile_width
    local_y = dy - tile_row * tile_height

    # --- SURVEY UNIT INDEX ---
    su_col = int(local_x // su_size)
    su_row = int(local_y // su_size)

    # Flip Y so origin is top-left (matches your examples)
    su_row_from_top = (tile_grid_y - 1) - su_row

    # IMPORTANT: numbering appears column-major in your expected output
    su = int(su_col * tile_grid_y + su_row_from_top + 1)

    # --- SUBCELL INDEX ---
    subcell_size = su_size / subcell_grid

    sc_local_x = local_x - su_col * su_size
    sc_local_y = local_y - su_row * su_size

    sc_col = int(sc_local_x // subcell_size)
    sc_row = int(sc_local_y // subcell_size)

    # Subcell numbering (row-major is typical; adjust if needed)
    sc = int(sc_row * subcell_grid + sc_col + 1)

    return {
        'tile': tile,
        'survey_unit': su,
        'subcell': sc
    }
$$;

In [ ]:
%%sql -r dataframe_1
-- I will put my prompt into the folder for this assignment. Below I had claude do the ranking and stability score.
-- ============================================================
-- 1. LOCAL RANK: Rank subcells within their Survey Unit
--    and Survey Units within their Tile
-- ============================================================
CREATE OR REPLACE VIEW LOCAL_RANK_VIEW AS
SELECT
    EASTING,
    NORTHING,
    READING,
    CONVERT_XY(
        EASTING, NORTHING,
        2180160.001, 6660000.000,
        32.81,
        21, 18,
        10
    ) AS grid,
    grid['tile']::STRING          AS tile,
    grid['survey_unit']::INTEGER  AS survey_unit,
    grid['subcell']::INTEGER      AS subcell,

    RANK() OVER (
        PARTITION BY grid['tile']::STRING, grid['survey_unit']::INTEGER
        ORDER BY READING DESC
    ) AS subcell_rank_in_su,

    RANK() OVER (
        PARTITION BY grid['tile']::STRING
        ORDER BY READING DESC
    ) AS su_rank_in_tile

FROM data5035.spring26.sdg_001_ra226_scandata;

In [ ]:
%%sql -r dataframe_12
#Here is the output from Claude. 
-- See the "hottest" subcells first
SELECT *
FROM LOCAL_RANK_VIEW
ORDER BY subcell_rank_in_su ASC
LIMIT 10;

-- See the most stable survey units first
SELECT *
FROM STABILITY_SCORE_VIEW
ORDER BY stability_score DESC
LIMIT 10;

In [ ]:
%%sql -r Combine
CREATE OR REPLACE VIEW STABILITY_SCORE_VIEW AS
SELECT
    grid['tile']::STRING         AS tile,
    grid['survey_unit']::INTEGER AS survey_unit,
    COUNT(*)                     AS reading_count,
    AVG(READING)                 AS avg_reading,
    STDDEV(READING)              AS stddev_reading,

--This plugs in the stability score formulas requested in the assignment. 
    1.0 / (1.0 + STDDEV(READING)) AS stability_score,
    STDDEV(READING) / NULLIF(AVG(READING), 0) AS sensor_variability_index

FROM (SELECT READING, CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata)
GROUP BY --this was reccomended to add in for the order. 
    grid['tile']::STRING,
    grid['survey_unit']::INTEGER;

In [ ]:
%%sql -r Assess
CREATE OR REPLACE VIEW ZSCORE_VIEW AS
WITH global_stats AS (
    -- Calculate global mean and stddev across the entire dataset
    SELECT
        AVG(READING)    AS global_avg,
        STDDEV(READING) AS global_stddev
    FROM data5035.spring26.sdg_001_ra226_scandata
),
tile_stats AS (
    -- reccomended to use this method
    SELECT
        grid['tile']::STRING AS tile,
        AVG(READING)         AS tile_avg,
        COUNT(*)             AS reading_count
    FROM (
        SELECT
            READING,
            CONVERT_XY(
                EASTING, NORTHING,
                2180160.001, 6660000.000,
                32.81,
                21, 18,
                10) AS grid
        FROM data5035.spring26.sdg_001_ra226_scandata)
    GROUP BY grid['tile']::STRING)
SELECT
    t.tile,
    t.tile_avg,
    t.reading_count,
    g.global_avg,
    g.global_stddev,

    -- usage of the zscore formula per request of the functionality. 
    (t.tile_avg - g.global_avg) / NULLIF(g.global_stddev, 0) AS z_score

FROM tile_stats t
CROSS JOIN global_stats g;

In [ ]:
%%sql -r dataframe_15
SELECT *
FROM ZSCORE_VIEW
ORDER BY z_score DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_16
SELECT *
FROM ZSCORE_VIEW
ORDER BY z_score DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_17
SELECT *
FROM STABILITY_SCORE_VIEW
ORDER BY survey_unit DESC
LIMIT 10;

In [ ]:
%%sql -r Assess
CREATE OR REPLACE VIEW COMBINED_SURVEY_VIEW AS
WITH base AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata
),
grid_labeled AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        grid['tile']::STRING         AS tile,
        grid['survey_unit']::INTEGER AS survey_unit,
        grid['subcell']::INTEGER     AS subcell
    FROM base
),
global_stats AS (
    SELECT
        AVG(READING)    AS global_avg,
        STDDEV(READING) AS global_stddev
    FROM data5035.spring26.sdg_001_ra226_scandata
),
tile_stats AS (
    SELECT
        tile,
        AVG(READING)   AS tile_avg,
        MAX(READING)   AS tile_max,
        COUNT(*)            AS reading_count
    FROM grid_labeled
    GROUP BY tile
)
SELECT
    g.EASTING,
    g.NORTHING,
    g.READING,
    g.tile,
    g.survey_unit,
    g.subcell,

    -- Local Rank: subcell within survey unit
    RANK() OVER (
        PARTITION BY g.tile, g.survey_unit
        ORDER BY g.READING DESC
    ) AS subcell_rank_in_su,

    -- Local Rank: survey unit within tile
    RANK() OVER (
        PARTITION BY g.tile
        ORDER BY g.READING DESC
    ) AS su_rank_in_tile,

    -- Stability Score
    1.0 / (1.0 + STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit)) AS stability_score,

    -- Sensor Variability Index
    STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit) /
        NULLIF(AVG(g.READING) OVER (PARTITION BY g.tile, g.survey_unit), 0) AS sensor_variability_index,

    -- Z-Score vs Global Mean
    (t.tile_avg - gs.global_avg) / NULLIF(gs.global_stddev, 0) AS z_score,

    -- Reference Range Status
    CASE
        WHEN g.READING < 5   THEN 'OK'
        WHEN g.READING < 7.4 THEN 'Warning'
        ELSE                           'Alarm'
    END AS reference_range_status,

    -- Layered Averages
    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit, g.subcell
    ) AS avg_sensor_subcell,

    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit
    ) AS avg_sensor_survey_unit,

    AVG(g.READING) OVER (
        PARTITION BY g.tile
    ) AS avg_sensor_tile,

    -- ============================================================
    -- PERCENTILE: Relative severity across the dataset
    -- ============================================================

    -- Percentile rank based on average sensor value per tile
    PERCENT_RANK() OVER (
        ORDER BY t.tile_avg ASC
    ) AS percentile_avg_sensor,

    -- Percentile rank based on max sensor value per tile
    PERCENT_RANK() OVER (
        ORDER BY t.tile_max ASC
    ) AS percentile_max_sensor

FROM grid_labeled g
CROSS JOIN global_stats gs
JOIN tile_stats t ON g.tile = t.tile;

--had an app reorganize the code. I need to double check to make sure this hasn't been messed up

In [ ]:
%%sql -r dataframe_19
SELECT *
FROM COMBINED_SURVEY_VIEW ORDER BY percentile_max_sensor DESC
LIMIT 10;

In [ ]:
%%sql -r dataframe_8
CREATE OR REPLACE VIEW COMBINED_SURVEY_VIEW AS
WITH base AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata
),
grid_labeled AS (
    SELECT
        EASTING,
        NORTHING,
        READING,
        grid['tile']::STRING         AS tile,
        grid['survey_unit']::INTEGER AS survey_unit,
        grid['subcell']::INTEGER     AS subcell
    FROM base
),
full_stats AS (
    SELECT
        AVG(READING)    AS full_avg,
        STDDEV(READING) AS full_stddev
    FROM data5035.spring26.sdg_001_ra226_scandata
),
tile_stats AS (
    SELECT
        tile,
        AVG(READING)   AS tile_avg,
        MAX(READING)   AS tile_max,
        COUNT(*)            AS reading_count
    FROM grid_labeled
    GROUP BY tile
),
boundary AS (
    SELECT
        MIN(EASTING)  AS min_easting,
        MAX(EASTING)  AS max_easting,
        MIN(NORTHING) AS min_northing,
        MAX(NORTHING) AS max_northing
    FROM data5035.spring26.sdg_001_ra226_scandata
),
subcell_bounds AS ( --this was added to create the boundaries of the subcell. The math was a little interesting. I undertood the max and min and had the math done with help. 
    SELECT
        min_easting,
        max_easting,
        min_northing,
        max_northing,
        32.81 / 10 AS subcell_size
    FROM boundary
)
SELECT
    g.EASTING,
    g.NORTHING,
    g.READING,
    g.tile,
    g.survey_unit,
    g.subcell,

    RANK() OVER (
        PARTITION BY g.tile, g.survey_unit
        ORDER BY g.READING DESC
    ) AS subcell_rank_in_su,

    RANK() OVER (
        PARTITION BY g.tile
        ORDER BY g.READING DESC
    ) AS su_rank_in_tile,

    1.0 / (1.0 + STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit)) AS stability_score,

    STDDEV(g.READING) OVER (PARTITION BY g.tile, g.survey_unit) /
        NULLIF(AVG(g.READING) OVER (PARTITION BY g.tile, g.survey_unit), 0) AS sensor_variability_index,

    (t.tile_avg - gs.full_avg) / NULLIF(gs.full_stddev, 0) AS z_score,

    CASE
        WHEN g.READING < 5   THEN 'OK'
        WHEN g.READING < 7.4 THEN 'Warning' --math is from the prompt
        ELSE                           'Alarm'
    END AS reference_range_status,

    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit, g.subcell
    ) AS avg_sensor_subcell,

    AVG(g.READING) OVER (
        PARTITION BY g.tile, g.survey_unit
    ) AS avg_sensor_survey_unit,

    AVG(g.READING) OVER (
        PARTITION BY g.tile
    ) AS avg_sensor_tile,

    PERCENT_RANK() OVER (
        ORDER BY t.tile_avg ASC
    ) AS percentile_avg_sensor,

    PERCENT_RANK() OVER (
        ORDER BY t.tile_max ASC
    ) AS percentile_max_sensor,
--they are building a case here wtih the precent ranka nd after foing the ranking they are comparing the max and min to figure out where it maps. 
    CASE
        WHEN g.EASTING  <= sb.min_easting  + sb.subcell_size THEN TRUE
        WHEN g.EASTING  >= sb.max_easting  - sb.subcell_size THEN TRUE
        WHEN g.NORTHING <= sb.min_northing + sb.subcell_size THEN TRUE
        WHEN g.NORTHING >= sb.max_northing - sb.subcell_size THEN TRUE
        ELSE FALSE
    END AS is_edge_subcell

FROM grid_labeled g
CROSS JOIN full_stats gs
CROSS JOIN subcell_bounds sb
JOIN tile_stats t ON g.tile = t.tile;

In [ ]:
%%sql -r dataframe_10
-- See all edge subcells
SELECT *
FROM COMBINED_SURVEY_VIEW
WHERE is_edge_subcell = TRUE
LIMIT 10;

-- Get a count of edge vs non-edge
SELECT
    is_edge_subcell,
    COUNT(*) AS reading_count
FROM COMBINED_SURVEY_VIEW
GROUP BY is_edge_subcell;

In [ ]:
%%sql -r convert_xy_test
SELECT CONVERT_XY(
    2180160.0001,
    6660000.0000,
    2180160.001,
    6660000.000,
    32.81,
    21,
    18,
    10
);

In [ ]:
%%sql -r stability_score_result
-- ============================================================
-- 2. STABILITY SCORE: 1 / (1 + STDDEV) at the Survey Unit level
--    Higher score = more stable/consistent readings
-- ============================================================
CREATE OR REPLACE VIEW STABILITY_SCORE_VIEW AS
SELECT
    grid['tile']::STRING         AS tile,
    grid['survey_unit']::INTEGER AS survey_unit,
    COUNT(*)                     AS reading_count,
    AVG(READING)                 AS avg_reading,
    STDDEV(READING)              AS stddev_reading,
    1.0 / (1.0 + STDDEV(READING)) AS stability_score
FROM (
    SELECT
        READING,
        CONVERT_XY(
            EASTING, NORTHING,
            2180160.001, 6660000.000,
            32.81,
            21, 18,
            10
        ) AS grid
    FROM data5035.spring26.sdg_001_ra226_scandata
)
GROUP BY
    grid['tile']::STRING,
    grid['survey_unit']::INTEGER; --used Claude for this code and as a template. 